# EMR Event-Prediction — Optimisation Analysis

Two views of the optimisation effort on MIMIC-IV:

1. **Optimisation journey** (`results/results-architecture optimization.tsv`) — every experiment in chronological order, with the running best AUROC / AUPRC / MAE.
2. **Architecture size comparison** (`results/results-hyperparameters sweep.tsv`) — the final grid of architecture sizes evaluated against the locked baseline.

Narrative report: `status.md`. Final checkpoints: `emr_model/checkpoints/`. Final best model: **M-256** (commit `5496c9e`).

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.2,
})

## 1. Optimisation journey

Every experiment in chronological order. Discarded experiments are shown in light grey, kept ones in green; the running best line tracks the locked-in improvement over time.

In [ ]:
trend = pd.read_csv('results/results-architecture optimization.tsv', sep='\t')
for col in ['outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb']:
    trend[col] = pd.to_numeric(trend[col], errors='coerce')
trend['status'] = trend['status'].str.strip().str.upper()

print(f'Total experiments: {len(trend)}')
print(trend['status'].value_counts().to_string())
trend.head()

In [ ]:
valid = trend[trend['status'] != 'CRASH'].copy().reset_index(drop=True)
kept_mask = valid['status'] == 'KEEP'
kept_idx  = valid.index[kept_mask]
kept      = valid[kept_mask]
disc      = valid[valid['status'] == 'DISCARD']

fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

# Panel 1: AUROC (primary, non-decreasing running best)
ax = axes[0]
ax.axhline(0.5, color='#aaaaaa', linewidth=1, linestyle='--', label='Random (0.5)')
ax.scatter(disc.index, disc['outcome_auroc'], c='#cccccc', s=14, alpha=0.5, zorder=2, label='Discarded')
ax.scatter(kept_idx,   kept['outcome_auroc'], c='#2ecc71', s=55, zorder=4,
           label='Kept', edgecolors='black', linewidths=0.5)
ax.step(kept_idx, kept['outcome_auroc'].cummax(), where='post',
        color='#27ae60', linewidth=2, alpha=0.7, zorder=3, label='Running best')
ax.set_ylabel('Outcome AUROC  \u2191', fontsize=11)
ax.set_title(f'Optimisation journey \u2014 {len(trend)} experiments, {kept_mask.sum()} kept',
             fontsize=13, loc='left')
ax.legend(loc='lower right', fontsize=9)

# Panel 2: AUPRC (secondary, non-decreasing)
ax = axes[1]
ax.scatter(disc.index, disc['outcome_auprc'], c='#cccccc', s=14, alpha=0.5, zorder=2, label='Discarded')
ax.scatter(kept_idx,   kept['outcome_auprc'], c='#9b59b6', s=55, zorder=4,
           label='Kept', edgecolors='black', linewidths=0.5)
ax.step(kept_idx, kept['outcome_auprc'].cummax(), where='post',
        color='#8e44ad', linewidth=2, alpha=0.7, zorder=3, label='Running best')
ax.set_ylabel('Outcome AUPRC  \u2191', fontsize=11)
ax.legend(loc='lower right', fontsize=9)

# Panel 3: Onset MAE (tertiary, non-increasing)
ax = axes[2]
ax.scatter(disc.index, disc['onset_mae_hrs'], c='#cccccc', s=14, alpha=0.5, zorder=2, label='Discarded')
ax.scatter(kept_idx,   kept['onset_mae_hrs'], c='#e67e22', s=55, zorder=4,
           label='Kept', edgecolors='black', linewidths=0.5)
ax.step(kept_idx, kept['onset_mae_hrs'].cummin(), where='post',
        color='#d35400', linewidth=2, alpha=0.7, zorder=3, label='Running best')
ax.set_xlabel('Experiment #', fontsize=11)
ax.set_ylabel('Onset MAE (h)  \u2193', fontsize=11)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

## 2. Architecture size comparison

Final grid of architecture sizes evaluated against the locked M-256 baseline. The dedupe step picks the canonical row per architecture: M-256 uses the retrain (the one whose checkpoints we deployed); L-384 uses the within-size tuned final; M-256-QA uses the properly-tokenized run.

In [ ]:
df = pd.read_csv('results/results-hyperparameters sweep.tsv', sep='\t')
for col in ['outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Anchor on the leading 'arch <name>' prefix so comparison phrases mid-description
# (e.g. 'vs M-256;') don't false-match.
_ARCH_RE = re.compile(r'^\s*arch\s+([A-Za-z0-9\-]+)')
_RENAMES = {
    'M-256-QA-proper': 'M-256-QA',
    'M-256-QA':        'M-256-QA (stale tokenizer)',
    'L-384-v2':        'L-384',
    'L-384':           'L-384 (initial)',
}

def tag(desc):
    m = _ARCH_RE.match(str(desc))
    if not m:
        return str(desc)[:20]
    raw = m.group(1)
    if 'retrain' in str(desc):
        return 'M-256'
    return _RENAMES.get(raw, raw)

df['arch'] = df['description'].apply(tag)
df[['commit', 'arch', 'outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb', 'status']]

In [ ]:
final_order = ['S-128', 'M-256', 'M-256-deep', 'L-384', 'M-256-QA']
present     = [a for a in final_order if a in set(df['arch'])]
missing     = [a for a in final_order if a not in set(df['arch'])]
if missing:
    print(f'[note] architectures not present in this TSV \u2014 skipped: {missing}')

final = (
    df[df['arch'].isin(present)]
    .drop_duplicates('arch', keep='last')
    .set_index('arch')
    .loc[present]
)
final[['commit', 'outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb', 'status']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
labels = final.index.tolist()
best   = final['outcome_auroc'].idxmax()
colors = ['#2ecc71' if a == best else '#7f8c8d' for a in labels]

for ax, col, ylabel, fmt in [
    (axes[0], 'outcome_auroc', 'AUROC  \u2191',         '{:.3f}'),
    (axes[1], 'outcome_auprc', 'AUPRC  \u2191',         '{:.3f}'),
    (axes[2], 'onset_mae_hrs', 'Onset MAE (h)  \u2193', '{:.2f}'),
]:
    bars = ax.bar(labels, final[col], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(ylabel, fontsize=11)
    for b, v in zip(bars, final[col]):
        ax.text(b.get_x() + b.get_width() / 2, v, ' ' + fmt.format(v),
                ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', rotation=20)

axes[0].set_title(f'Architecture sweep \u2014 best: {best} (highlighted)', fontsize=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
param_counts = {
    'S-128': 1.67, 'M-256': 6.41, 'M-256-deep': 9.31, 'L-384': 20.78, 'M-256-QA': 6.42,
}
final['params_M'] = final.index.map(param_counts)

fig, ax = plt.subplots(figsize=(8, 5))
for arch, row in final.iterrows():
    if pd.isna(row['params_M']):
        continue
    c = '#2ecc71' if arch == best else '#7f8c8d'
    ax.scatter(row['params_M'], row['outcome_auroc'],
               s=160, c=c, edgecolor='black', linewidth=0.6, zorder=3)
    ax.annotate(arch, (row['params_M'], row['outcome_auroc']),
                textcoords='offset points', xytext=(8, 6), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('Parameters (millions, log scale)')
ax.set_ylabel('AUROC')
ax.set_title('Capacity vs accuracy \u2014 predictive signal saturates at M-256',
             fontsize=11, loc='left')
plt.tight_layout()
plt.show()

## Final summary

See `status.md` for the full narrative report, per-outcome breakdown, k-day scan, and reproducibility table.

In [ ]:
best_row = final.loc[best]
print(f'Final best:        {best}')
print(f'  commit:          {best_row["commit"]}')
print(f'  AUROC:           {best_row["outcome_auroc"]:.4f}')
print(f'  AUPRC:           {best_row["outcome_auprc"]:.4f}')
print(f'  Onset MAE (h):   {best_row["onset_mae_hrs"]:.2f}')
print(f'  Peak VRAM (GB):  {best_row["peak_vram_gb"]:.2f}')
print(f'  Parameters:      {param_counts[best]:.2f} M')